# 02 - Data Preprocessing

Week 3 deliverable: clean the selected non-agent features, encode categorical variables, normalize numeric variables, and create a time-based train/test split.

Default split used here:
- **Test set:** May 2026, the most recent month available.
- **Training set:** February, March, and April 2026.
- **Training window X:** 3 months, configurable below.

## Preprocessing Decisions

- Restrict to `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.
- Drop records with missing date or missing latitude/longitude.
- Keep close prices between `$10,000` and `$20,000,000` to remove obvious bad records and extreme outliers.
- Impute numeric missing values with **training-set medians**.
- Convert boolean fields to `0/1`, treating missing values as `0`.
- Convert categorical location fields to numeric using **training-set frequency encoding**.
- Normalize numeric features using **training-set mean/std**.
- Add `HomeAge = 2026 - YearBuilt`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
DATA_FILE = Path("CRMLSSold_combined.csv")
OUTPUT_FILE = Path("week3_cleaned_model_data.csv")
WINDOW_RESULTS_FILE = Path("week3_training_window_results.csv")

CURRENT_YEAR = 2026
MIN_CLOSE_PRICE = 10_000
MAX_CLOSE_PRICE = 20_000_000

TEST_MONTH = "2026-05"
TRAIN_MONTHS = 3

location_features = [
    "Latitude",
    "Longitude",
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
]

property_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "GarageSpaces",
    "ParkingTotal",
]

target = "ClosePrice"
id_and_date_cols = ["ListingId", "CloseDate"]
filter_cols = ["PropertyType", "PropertySubType"]
selected_cols = id_and_date_cols + filter_cols + location_features + property_features + [target]


## Load Data

In [ ]:
df = pd.read_csv(DATA_FILE, dtype=str, keep_default_na=False, low_memory=False)
df.shape


## Base Cleaning

In [ ]:
def clean_base(raw_df: pd.DataFrame) -> pd.DataFrame:
    work = raw_df[selected_cols].copy()

    work = work[
        (work["PropertyType"] == "Residential")
        & (work["PropertySubType"] == "SingleFamilyResidence")
    ].copy()

    work["CloseDate"] = pd.to_datetime(work["CloseDate"], errors="coerce")
    work["close_month"] = work["CloseDate"].dt.to_period("M").astype(str)

    numeric_cols = [
        "Latitude",
        "Longitude",
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "YearBuilt",
        "Stories",
        "GarageSpaces",
        "ParkingTotal",
        target,
    ]

    for col in numeric_cols:
        work[col] = pd.to_numeric(work[col].replace("", np.nan), errors="coerce")

    work = work[(work[target] >= MIN_CLOSE_PRICE) & (work[target] <= MAX_CLOSE_PRICE)].copy()
    work = work.dropna(subset=["CloseDate", "Latitude", "Longitude"]).copy()

    for col in ["ViewYN", "PoolPrivateYN", "AttachedGarageYN"]:
        normalized = work[col].astype(str).str.strip().str.lower()
        work[col] = (
            normalized
            .map({"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0})
            .fillna(0)
            .astype(int)
        )

    for col in ["City", "PostalCode", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict"]:
        work[col] = work[col].astype(str).str.strip().replace("", "Unknown")

    work["PostalCode"] = work["PostalCode"].str.extract(r"(\d{5})", expand=False).fillna("Unknown")
    work["HomeAge"] = CURRENT_YEAR - work["YearBuilt"]
    work.loc[(work["HomeAge"] < 0) | (work["HomeAge"] > 250), "HomeAge"] = np.nan

    return work


base = clean_base(df)

print(f"Raw rows: {len(df):,}")
print(f"Rows after base cleaning: {len(base):,}")
base["close_month"].value_counts().sort_index()


## Time-Based Split

In [ ]:
def month_window(test_month: str, train_months: int) -> list[str]:
    test_period = pd.Period(test_month, freq="M")
    return [(test_period - i).strftime("%Y-%m") for i in range(train_months, 0, -1)]


train_month_list = month_window(TEST_MONTH, TRAIN_MONTHS)
train_month_list, TEST_MONTH


## Fit Preprocessing On Train, Apply To Train/Test

In [ ]:
def build_model_data(base_df: pd.DataFrame, train_months: int, test_month: str):
    train_month_list = month_window(test_month, train_months)
    keep_months = set(train_month_list + [test_month])

    data = base_df[base_df["close_month"].isin(keep_months)].copy()
    data["split"] = np.where(data["close_month"] == test_month, "test", "train")
    train_mask = data["split"].eq("train")

    numeric_features = [
        "Latitude",
        "Longitude",
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "YearBuilt",
        "Stories",
        "GarageSpaces",
        "ParkingTotal",
        "HomeAge",
    ]
    bool_features = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN"]
    categorical_features = ["City", "PostalCode", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict"]

    impute_values = data.loc[train_mask, numeric_features].median()
    data[numeric_features] = data[numeric_features].fillna(impute_values)

    scale_mean = data.loc[train_mask, numeric_features].mean()
    scale_std = data.loc[train_mask, numeric_features].std().replace(0, 1)
    for col in numeric_features:
        data[f"{col}_scaled"] = (data[col] - scale_mean[col]) / scale_std[col]

    for col in categorical_features:
        train_frequencies = data.loc[train_mask, col].value_counts(normalize=True)
        data[f"{col}_freq"] = data[col].map(train_frequencies).fillna(0.0)

    data["LogClosePrice"] = np.log1p(data[target])

    feature_cols = (
        [f"{col}_scaled" for col in numeric_features]
        + bool_features
        + [f"{col}_freq" for col in categorical_features]
    )
    output_cols = ["ListingId", "CloseDate", "close_month", "split"] + feature_cols + [target, "LogClosePrice"]

    metadata = {
        "train_months": train_month_list,
        "test_month": test_month,
        "feature_cols": feature_cols,
        "impute_values": impute_values,
        "scale_mean": scale_mean,
        "scale_std": scale_std,
    }

    return data[output_cols].copy(), metadata


model_data, metadata = build_model_data(base, TRAIN_MONTHS, TEST_MONTH)

print(f"Train months: {metadata['train_months']}")
print(f"Test month: {metadata['test_month']}")
print(model_data["split"].value_counts())
print(f"Missing values: {model_data.isna().sum().sum():,}")
model_data.head()


## Save Cleaned CSV

In [ ]:
model_data.to_csv(OUTPUT_FILE, index=False)

print(f"Wrote {OUTPUT_FILE}")
print(f"Shape: {model_data.shape}")
print(f"Train rows: {(model_data['split'] == 'train').sum():,}")
print(f"Test rows: {(model_data['split'] == 'test').sum():,}")


## Experiment With Training Window Length X

Week 3 says X is tunable. This quick baseline compares `X = 1..12` months using May 2026 as the fixed test set. The model is intentionally lightweight; use this to choose a reasonable window before doing deeper model tuning.

In [ ]:
window_results = []

for x in range(1, 13):
    candidate_data, candidate_metadata = build_model_data(base, x, TEST_MONTH)
    train = candidate_data[candidate_data["split"] == "train"]
    test = candidate_data[candidate_data["split"] == "test"]

    if train.empty or test.empty:
        continue

    feature_cols = candidate_metadata["feature_cols"]
    model = HistGradientBoostingRegressor(
        max_iter=120,
        learning_rate=0.08,
        max_leaf_nodes=31,
        random_state=42,
    )
    model.fit(train[feature_cols], train["LogClosePrice"])

    pred = np.expm1(model.predict(test[feature_cols]))
    y = test[target].to_numpy()

    window_results.append({
        "train_months_x": x,
        "train_months": ",".join(candidate_metadata["train_months"]),
        "test_month": TEST_MONTH,
        "train_rows": len(train),
        "test_rows": len(test),
        "mae_close_price": mean_absolute_error(y, pred),
        "rmse_close_price": float(np.sqrt(mean_squared_error(y, pred))),
        "r2_close_price": r2_score(y, pred),
    })

window_results_df = pd.DataFrame(window_results).sort_values("mae_close_price")
window_results_df.to_csv(WINDOW_RESULTS_FILE, index=False)
window_results_df


Current run summary: `X = 3` is a reasonable default because Feb-Apr 2026 gives the best R² among the tested windows and nearly the same MAE as the longer windows. `X = 10` has the lowest MAE in this quick baseline, so it is worth revisiting during model tuning.